In [1]:
import re

def extract_video_id(youtube_url):
    """Extract video ID from various YouTube URL formats"""
    patterns = [
        r'(?:v=|\/)([0-9A-Za-z_-]{11}).*',
        r'(?:embed\/)([0-9A-Za-z_-]{11})',
        r'(?:shorts\/)([0-9A-Za-z_-]{11})'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, youtube_url)
        if match:
            return match.group(1)
    return None

In [2]:
SUPPORTED_LANGUAGES = {
    'en': 'English',
    'hi': 'Hindi',
    'hi-en': 'Hinglish (Hindi-English mix)',
    'es': 'Spanish',
    'fr': 'French'
}

def get_transcript_with_fallback(video_id, preferred_lang='en'):
    """Try multiple languages with fallback"""
    languages_to_try = [preferred_lang, 'en']  # Always fallback to English
    
    for lang in languages_to_try:
        try:
            transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=[lang])
            return " ".join(chunk["text"] for chunk in transcript_list), lang
        except:
            continue
    
    # If no captions, raise error
    raise Exception("No captions available in supported languages")

In [8]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFaceEndpoint
import os



# Embedding Model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Updated Endpoint Model for Hugging Face LLMs
llm = HuggingFaceEndpoint(
    repo_id="google/flan-t5-large",
    task="text2text-generation"
)


In [9]:
# Install: pip install streamlit
import streamlit as st

def main():
    st.title("🎬 YouTube Video Q&A Assistant")
    st.write("Get answers from any YouTube video!")
    
    # User inputs
    youtube_url = st.text_input("Paste YouTube URL:")
    language = st.selectbox("Select Language", list(SUPPORTED_LANGUAGES.keys()))
    question = st.text_input("Your question:")
    
    if st.button("Get Answer"):
        if youtube_url and question:
            with st.spinner("Processing video..."):
                try:
                    video_id = extract_video_id(youtube_url)
                    transcript, detected_lang = get_transcript_with_fallback(video_id, language)
                    
                    # Your existing QA chain here
                    answer = main_chain.invoke(question)
                    
                    st.success(f"Answer: {answer}")
                    st.info(f"Detected Language: {SUPPORTED_LANGUAGES[detected_lang]}")
                    
                except Exception as e:
                    st.error(f"Error: {str(e)}")

In [10]:
def generate_summary(transcript):
    """Generate video summary"""
    summary_prompt = PromptTemplate(
        template="""
        Summarize this video transcript in 3-5 key points:
        
        {context}
        
        Key Points:
        """,
        input_variables=['context']
    )
    
    summary_chain = (
        {"context": RunnablePassthrough()} 
        | summary_prompt 
        | llm 
        | StrOutputParser()
    )
    return summary_chain.invoke(transcript)

In [11]:
def detect_chapters(transcript):
    """Auto-detect video chapters/topics"""
    chapter_prompt = PromptTemplate(
        template="""
        Identify main topics/chapters from this transcript with timestamps:
        
        {context}
        
        Chapters:
        """,
        input_variables=['context']
    )
    
    chapter_chain = chapter_prompt | llm | StrOutputParser()
    return chapter_chain.invoke(transcript)

In [12]:
def get_key_takeaways(transcript):
    """Extract key learning points"""
    takeaways_prompt = PromptTemplate(
        template="""
        Extract 5-7 key learning points from this video:
        
        {context}
        
        Key Learnings:
        1.
        """,
        input_variables=['context']
    )
    
    takeaways_chain = takeaways_prompt | llm | StrOutputParser()
    return takeaways_chain.invoke(transcript)

In [16]:
import os
import re
import streamlit as st
from youtube_transcript_api import YouTubeTranscriptApi

# Text splitter (new package)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Hugging Face integrations (updated imports)
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFaceEndpoint

# Vector store and utilities
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Optional: ensure your HF token is set in the environment or .env
# os.environ["HUGGINGFACEHUB_API_TOKEN"] = "your_hf_token_here"

SUPPORTED_LANGUAGES = {"en": "English", "hi": "Hindi", "hi-en": "Hinglish"}


class YouTubeQASystem:
    def __init__(self, hf_embedding_model="sentence-transformers/all-MiniLM-L6-v2",
                 hf_llm_repo="google/flan-t5-large"):
        # Embeddings (Hugging Face)
        self.embeddings = HuggingFaceEmbeddings(model_name=hf_embedding_model)

        # LLM endpoint (Hugging Face Inference-like endpoint wrapper)
        # Use `parameters` for generation/runtime options (temperature, max_new_tokens, etc.)
        self.llm = HuggingFaceEndpoint(
            repo_id=hf_llm_repo,
            task="text2text-generation",
            parameters={"temperature": 0.2, "max_new_tokens": 512},
        )

        # retriever will be set after processing a video
        self.retriever = None

        # prompt template used to ask questions (simple, safe)
        self.prompt = PromptTemplate(
            template=(
                "Answer the question using ONLY the provided context from the video transcript.\n\n"
                "Context:\n{context}\n\n"
                "Question: {question}\n\n"
                "Answer:"
            ),
            input_variables=["context", "question"],
        )

        # chain placeholder (we'll build and call it dynamically in ask_question)
        self.parser = StrOutputParser()

    @staticmethod
    def extract_video_id(youtube_url: str) -> str | None:
        """Extract 11-char YouTube video id from a URL or return None."""
        pattern = r"(?:v=|\/)([0-9A-Za-z_-]{11})(?:\?|&|$)"
        m = re.search(pattern, youtube_url)
        return m.group(1) if m else None

    @staticmethod
    def get_transcript(video_id: str, preferred_lang: str = "en") -> tuple[str, str]:
        """Try to download transcript for preferred language, fallback to English."""
        for lang in (preferred_lang, "en"):
            try:
                transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=[lang])
                text = " ".join(chunk["text"] for chunk in transcript_list)
                return text, lang
            except Exception:
                continue
        raise RuntimeError("No captions available for this video.")

    def process_video(self, youtube_url: str, language: str = "en") -> tuple[str, str]:
        """
        Download transcript, split into chunks, build a FAISS vectorstore and retriever.
        Returns (transcript_text, detected_lang)
        """
        video_id = self.extract_video_id(youtube_url)
        if not video_id:
            raise ValueError("Invalid YouTube URL provided.")

        transcript, detected_lang = self.get_transcript(video_id, preferred_lang=language)

        # split transcript into document chunks
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        # create_documents is supported by this splitter to produce Document objects
        docs = splitter.create_documents([transcript])

        # create FAISS vectorstore from chunks
        vector_store = FAISS.from_documents(docs, embedding=self.embeddings)
        self.retriever = vector_store.as_retriever(search_kwargs={"k": 3})

        return transcript, detected_lang

    def ask_question(self, question: str) -> str:
        """Retrieve relevant chunks, format context, call the LLM and return the answer string."""
        if self.retriever is None:
            raise RuntimeError("No video processed. Call process_video() first.")

        # Retrieve relevant documents (API may be get_relevant_documents or get_relevant_documents)
        # Try common retrieval method names for compatibility:
        if hasattr(self.retriever, "get_relevant_documents"):
            docs = self.retriever.get_relevant_documents(question)
        elif hasattr(self.retriever, "get_relevant_items"):
            docs = self.retriever.get_relevant_items(question)
        else:
            # fallback: call the retriever as a callable
            docs = self.retriever(question)

        # format retrieved docs into context text
        context = "\n\n".join(getattr(d, "page_content", str(d)) for d in docs)

        # build chain on-the-fly: Prompt -> LLM -> Parser
        chain = self.prompt | self.llm | self.parser

        # invoke chain with context and question
        out = chain.invoke({"context": context, "question": question})
        return out


# Streamlit app
def main():
    st.set_page_config(page_title="YouTube Q&A Assistant", page_icon="🎬")
    st.title("🎬 YouTube Video Q&A Assistant")
    st.write("Ask questions about a YouTube video's transcript (English / Hindi / Hinglish).")

    if "qa_system" not in st.session_state:
        st.session_state.qa_system = YouTubeQASystem()

    with st.sidebar:
        st.header("⚡ Options")
        feature = st.selectbox("Feature", ["Q&A", "Summary", "Key Takeaways", "Chapters"])

    col1, col2 = st.columns([3, 1])
    with col1:
        youtube_url = st.text_input("YouTube URL", placeholder="https://www.youtube.com/watch?v=...")
    with col2:
        language = st.selectbox("Language", list(SUPPORTED_LANGUAGES.keys()),
                                format_func=lambda k: SUPPORTED_LANGUAGES[k])

    if youtube_url:
        try:
            if st.button("Process Video") or "transcript" not in st.session_state:
                with st.spinner("Processing transcript..."):
                    transcript, detected_lang = st.session_state.qa_system.process_video(youtube_url, language)
                    st.session_state.transcript = transcript
                    st.session_state.detected_lang = detected_lang

                st.success(f"Video processed (language: {SUPPORTED_LANGUAGES[detected_lang]})")
                with st.expander("Transcript Preview"):
                    st.text(transcript[:1000] + "..." if len(transcript) > 1000 else transcript)

            question = st.text_input("Ask a question about the video")
            if question and st.button("Get Answer"):
                with st.spinner("Answering..."):
                    answer = st.session_state.qa_system.ask_question(question)
                    st.success("Answer:")
                    st.write(answer)

        except Exception as exc:
            st.error(f"Error: {exc}")


if __name__ == "__main__":
    main()


2025-11-04 12:26:12.486 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-04 12:26:12.487 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-04 12:26:12.487 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-04 12:26:12.487 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-04 12:26:12.488 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-04 12:26:12.489 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-04 12:26:12.489 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-04 12:26:12.490 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar